# Personal Information
Full name: Vũ Duy Nhất

Student's ID: 24127095

Class: 24KHDL

**Note: For this assignment, you may only use standard Python and the `re` (Regular Expression) module. Advanced libraries such as NumPy, Pandas are not permitted**

## Exercise 1 — Phone numbers

Use `re.search` to find whether a string contains a phone number. The pattern that you write should detect a phone number in the following strings.  
```
"Call me at 382-384-3840."  
"my number is (510) 849-3519. Call me!"
```  
And not find a match in the following strings. 
```
"my number is 510-849-35192"  
"here’s my number: 510-849.3519"
``` 
Consider making your own tests as well  

In [1]:
import re

In [2]:
# YOUR CODE HERE
regex = r'\(?\d{3}\)?(-| )\d{3}-\d{4}(?!\d)'
text1 = "Call me at 382-384-3840."
text2 = "my number is (510) 849-3519. Call me!"
text3 = "my number is 510-849-35192"
text4 = "here’s my number: 510-849.3519"
print(re.search(regex, text1).group(0) == '382-384-3840') 
print(re.search(regex, text2).group(0) == '(510) 849-3519') 
print(re.search(regex, text3) == None)
print(re.search(regex, text4) == None)

True
True
True
True


## Exercise 2 — Normalizing dates

Use **one call** to `re.sub` to rewrite the string below so that every date uses a dash `-` between month, day and year.
```
03/12/2018, 03.13.18, 03/14/2018, 03:15:2018
```
Rules:
- A date is: 2-digit month, separator, 2-digit day, separator, 2- or 4-digit year. A separator is one of `/`, `.`, `:` or `-`.
- Change **only the separators inside dates**. Everything else (including the `, ` between dates) stays exactly the same.
- Keep the year as it is written: do **not** turn `18` into `2018`.

**Print:** the new string. It must be exactly:
```
03-12-2018, 03-13-18, 03-14-2018, 03-15-2018
```

In [3]:
# YOUR CODE HERE
text = '03/12/2018, 03.13.18, 03/14/2018, 03:15:2018'
regex = r'(?<!\d)(\d{2})[/.:-](\d{2})[/.:-](\d{4}|\d{2})(?!\d)'
print(re.sub(regex, r'\1-\2-\3', text))

03-12-2018, 03-13-18, 03-14-2018, 03-15-2018


## Exercise 3 — Extracting dialogue

Consider the first five sentences of the novel *Little Women* below. Extract the **spoken dialogue**: the text between a pair of double quotation marks `"…"`.

Rules:
- One string per quoted passage, in the order they appear.
- Do **not** include the quotation marks themselves.
- Keep everything inside the quotes exactly as written, including punctuation (e.g. the comma at the end of the first quote).
- Note that the last quote starts in the **middle** of its sentence.

**Print:** the list of extracted strings. It must contain **5** strings; the first one is
```
Christmas won't be Christmas without any presents,
```

In [4]:
text = '''
"Christmas won't be Christmas without any presents," grumbled Jo, lying on the rug.
"It's so dreadful to be poor!" sighed Meg, looking down at her old dress.
"I don't think it's fair for some girls to have plenty of pretty things, and other girls nothing at all," added little Amy, with an injured sniff.
"We've got Father and Mother, and each other," said Beth contentedly from her corner.
The four young faces on which the firelight shone brightened at the cheerful words, but darkened again as Jo said sadly, "We haven't got Father, and shall not have him for a long time."
'''
print(text)


"Christmas won't be Christmas without any presents," grumbled Jo, lying on the rug.
"It's so dreadful to be poor!" sighed Meg, looking down at her old dress.
"I don't think it's fair for some girls to have plenty of pretty things, and other girls nothing at all," added little Amy, with an injured sniff.
"We've got Father and Mother, and each other," said Beth contentedly from her corner.
The four young faces on which the firelight shone brightened at the cheerful words, but darkened again as Jo said sadly, "We haven't got Father, and shall not have him for a long time."



In [5]:
# YOUR CODE HERE
regex = r'"([\w\W]+?)"'
result = re.findall(regex, text)
for i in result:
    print(i)

Christmas won't be Christmas without any presents,
It's so dreadful to be poor!
I don't think it's fair for some girls to have plenty of pretty things, and other girls nothing at all,
We've got Father and Mother, and each other,
We haven't got Father, and shall not have him for a long time.


## Exercise 4 — Fraudulent emails

In this exercise you work with the file `email_test.txt` (attached): a collection of real scam emails.
Original dataset: https://www.kaggle.com/datasets/rtatman/fraudulent-email-corpus

### 4.0 Preparation (required — all later tasks use these rules)

1. **Read the file** with `open('email_test.txt', encoding='utf-8', errors='replace')`.
2. **Split it into emails.** Each email starts with a line that begins with `From r`, for example
   `From r  Wed Oct 30 21:41:56 2002`.
   This line is only a separator and is not part of the email.
   ⚠️ Do not confuse it with the header line `From: "MR. JAMES NGOLA." <james_ngola2002@maktoob.com>`.
   You should get **1,330 emails**.
3. **Header and body.** In each email, the **header** is everything before the first empty line; the **body** is everything after it.
4. **Header fields.** From the header, take the value of the **first** line that starts with `Subject:`, `From:` and `Date:` — the text after the colon, with leading and trailing spaces removed. If a field is missing, use the empty string `''` (some emails have no `Date:` or `From:` line).
5. **Decode the body.** Many bodies use *quoted-printable* encoding, which breaks words and replaces characters with codes. Clean **every** body with these two steps, in this order:
   1. Remove every *soft line break*: an `=` directly followed by a newline. Example: `neve=` + newline + `r` → `never`.
   2. Replace every `=` followed by **exactly two hexadecimal digits** (`0-9`, `A-F`, `a-f`) with the character that has that code, `chr(int(two_digits, 16))`. Examples: `=2C` → `,` `=2E` → `.` `=20` → space `=28` → `(`.
6. **Text of an email** = subject + `'\n'` + decoded body. Unless a task says otherwise, every search below is done in this text.

**Print:** the number of emails, and the subject of the first email (it should be `URGENT BUSINESS ASSISTANCE AND PARTNERSHIP`).

In [6]:
# YOUR CODE HERE: 4.0 preparation (read, split, parse, decode)
def read_file(path):
    with open(path, encoding='utf-8', errors='replace') as f:
        output = f.read()
    return output

def split_emails(raw_emails):
    regex = r'^From r[^\n]*(?:\n|$)'
    parts = re.split(regex, raw_emails, flags=re.MULTILINE)
    return [part for part in parts if part.strip()]

def parse_emails(splitted_emails):
    results = []

    for email in splitted_emails:
        header, _, body = email.partition('\n\n')
        result = {'Subject': '', 'From': '', 'Date': '', 'body': body}
        seen = set()

        for line in header.splitlines():
            for field in ('Subject', 'From', 'Date'):
                if line.startswith(field + ':') and field not in seen:
                    result[field] = line.split(':', 1)[1].strip()
                    seen.add(field)

        results.append(result)

    return results
                
def decode_body(parsed_emails):
    for i in parsed_emails:
        i['body'] = i['body'].replace('=\n', '')
        regex = '=([0-9A-Fa-f]{2})'
        i['body'] = re.sub(regex, lambda x:  chr(int(x.group(1), 16)), i['body'])
    return parsed_emails
        
        
        
        
raw_emails = read_file('email_test.txt')
splitted_emails = split_emails(raw_emails)
parsed_emails = parse_emails(splitted_emails)
decoded_emails = decode_body(parsed_emails)
processed_emails = [i['Subject'] + '\n' + i['body'] for i in decoded_emails]

print(f'The number of emails: {len(processed_emails)}')
print(f'The subject of first email: {processed_emails[0].splitlines()[0]}')

The number of emails: 1330
The subject of first email: URGENT BUSINESS ASSISTANCE AND PARTNERSHIP


### Simple fraudulent email detection

**4.1 Urgency words.** Count how many emails contain at least one **urgency word**:
`URGENT`, `IMMEDIATELY`, `QUICK`, `ASSISTANCE`, `CONFIDENTIAL`.

Rules:
- Case-insensitive: `urgent`, `Urgent` and `URGENT` all count.
- Whole words only: `URGENTLY`, `QUICKLY`, `CONFIDENTIALITY` do **not** count.

**Print:**
- the number of emails that contain at least one urgency word;
- the percentage of all emails this represents, rounded to 2 decimals (e.g. `12.34%`).

In [7]:
# YOUR CODE HERE

regex = r'\b(?:URGENT|IMMEDIATELY|QUICK|ASSISTANCE|CONFIDENTIAL)\b'

urgency_detects = [re.findall(regex, email, re.IGNORECASE) for email in processed_emails]
count = sum(1 for urgency in urgency_detects if urgency)

percentage = count / len(processed_emails) * 100

print(f"The number of emails that contain at least one urgency word: {count}")
print(f"Percentage: {percentage:.2f}%")

The number of emails that contain at least one urgency word: 1143
Percentage: 85.94%


**4.2 Money amounts.** Find all **money mentions** in the emails (e.g. `US$25M`, `$100,000.00`, `USD$31,000,000.00`).

A money mention is: **currency marker + number + optional multiplier**.

| Part | Rule |
|---|---|
| Currency marker | `US$`, `USD$`, `USD` or `$` (any case), written **before** the number, followed by at most one space |
| Number | starts and ends with a digit; may contain `,` and `.` in between. Examples: `25`, `21.5`, `100,000.00` |
| Multiplier (optional) | `M`, `MILLION`, `B` or `BILLION` (any case), after at most one space, and **not directly followed by another letter** (`$12m` has multiplier `m`; in `$29mUSD` the `m` is **not** a multiplier) |

Amounts written in words, or with the currency **after** the number (`500 USD`), are not money mentions.

**Converting a mention to a value (in dollars):**
1. Remove all commas.
2. If more than one `.` remains, keep only the **last** one: `126000.000.00` → `126000000.00`.
3. Convert to a `float`.
4. If the value is **below 1,000**, multiply it by 1,000,000 for `M`/`MILLION` or by 1,000,000,000 for `B`/`BILLION`. If the value is **1,000 or more**, ignore the multiplier: scammers often write `$12,500,000.00 Million` when they mean 12.5 million.

| Mention | Value |
|---|---|
| `US$25M` | 25,000,000 |
| `$100,000.00` | 100,000 |
| `USD$31,000,000.00` | 31,000,000 |
| `US$21.5 MILLION` | 21,500,000 |
| `$12,500,000.00 Million` | 12,500,000 |

**Print** (amounts with thousands separators and 2 decimals, e.g. `25,000,000.00`):
- the total number of money mentions across all emails;
- the largest amount;
- the smallest amount **greater than 0**;
- the average amount per email: for each email with at least one mention, take the **largest** amount in that email; print the average of these values, and how many emails were used.

In [8]:
# YOUR CODE HERE
def parse_money(mention):
    regex = (
        r'(?:USD\$|US\$|USD|\$) ?'
        r'(\d(?:[\d,.]*\d)?)'
        r'(?: ?(MILLION|BILLION|M|B)(?![A-Za-z]))?'
    )
    match = re.fullmatch(regex, mention, re.IGNORECASE)


    number = match.group(1)
    multiplier = (match.group(2) or '').upper()

    number = number.replace(',', '')

    if number.count('.') > 1:
        parts = number.split('.')
        number = ''.join(parts[:-1]) + '.' + parts[-1]

    value = float(number)

    if value < 1000:
        if multiplier in ('M', 'MILLION'):
            value *= 1_000_000
        elif multiplier in ('B', 'BILLION'):
            value *= 1_000_000_000

    return value
def total_num_money_mentions(money_mentions):
    result = 0
    for email_mention in money_mentions:
        if email_mention:
            result += len(email_mention)
    return result

def max_money_mentions(money_mentions):
    result = None
    for email_mention in money_mentions:
        if result != None and email_mention:
            result = max(result, max(email_mention))
        elif email_mention:
            result = max(email_mention)
    return result

def min_money_mentions(money_mentions):
    result = None

    for email_mention in money_mentions:
        for value in email_mention:
            if value > 0:
                if result is None:
                    result = value
                else:
                    result = min(result, value)

    return result

def avr_money_mentions(money_mentions):
    result = 0
    at_least_1_count = 0
    for email_mention in money_mentions:
        if email_mention:
            result += max(email_mention)
            at_least_1_count += 1
    return result/at_least_1_count , at_least_1_count

regex = (
    r'((?:USD\$|US\$|USD|\$) ?'
    r'\d(?:[\d,.]*\d)?'
    r'(?: ?(?:MILLION|BILLION|M|B)(?![A-Za-z]))?)'
)
money_mentions = [re.findall(regex, email, re.IGNORECASE) for email in processed_emails]

for i in range(len(money_mentions)):
    if money_mentions[i]:
        money_mentions[i] = [parse_money(j) for j in money_mentions[i]]
avr_mentioned_money, emails_used = avr_money_mentions(money_mentions)
print (f'The total number of money mentions: {total_num_money_mentions(money_mentions)}')
print (f'The largest amount of mentioned money: {max_money_mentions(money_mentions):,.2f}')
print (f'The smallest amount of mentioned money: {min_money_mentions(money_mentions):,.2f}')
print (f'The average amount of mentioned money: {avr_mentioned_money:,.2f}')
print (f'The number of emails used: {emails_used}')

The total number of money mentions: 2120
The largest amount of mentioned money: 55,000,000,000.00
The smallest amount of mentioned money: 0.28
The average amount of mentioned money: 201,262,878.40
The number of emails used: 1112


**4.3 Death stories.** Extract all mentions of death or deceased persons.

A death mention is any of the following (case-insensitive, whole words only):
- the words `died`, `dead`, `deceased`, `death`, `killed`, `murdered`, `assassinated`, `demise`;
- the phrase `passed away`;
- the word `late` followed by one of: `husband`, `wife`, `father`, `mother`, `parents`, `son`, `daughter`, `brother`, `sister`, `uncle`, `client`, `customer`, `president`, `mr`, `mrs`, `dr` — e.g. `late father`, `LATE PRESIDENT`, `late Mr`.

In the two-word phrases, the words may be separated by any whitespace (spaces or a line break).

**Print:**
- the total number of death mentions;
- the number of emails with at least one death mention, and their percentage of all emails (2 decimals);
- the 5 most frequent mentions with their counts. Before counting, write each mention in lowercase with a single space between words (so `LATE  Father` counts as `late father`).

In [9]:
# YOUR CODE HERE
def total_death_mentions(death_stories):
    result = 0
    for death_story in death_stories:
        if death_story:
            result += len(death_story)
    return result

def num_email_have_death_mention(death_stories):
    count = 0
    for death_story in death_stories:
        if death_story:
            count += 1
    return count

def top_5_death_mentions(death_stories):
    counts = {}

    for death_story in death_stories:
        if death_story:
            for mention in death_story:
                normalized = ' '.join(mention.lower().split())
                counts[normalized] = counts.get(normalized, 0) + 1

    return sorted(counts.items(), key=lambda item: item[1], reverse=True)[:5]

regex_1 = r'\b(?:died|dead|deceased|death|killed|murdered|assassinated|demise)\b'
regex_2 = r'\bpassed\s+away\b'
regex_3 = (
    r'\blate\s+'
    r'(?:husband|wife|father|mother|parents|son|daughter|'
    r'brother|sister|uncle|client|customer|president|mr|mrs|dr)\b'
)
regex = '|'.join([regex_1, regex_2, regex_3])
death_stories = [re.findall(regex, email, re.IGNORECASE) for email in processed_emails]

print(f'The total number of death mentions: {total_death_mentions(death_stories)}')
count_email_at_least_1 = num_email_have_death_mention(death_stories)
print(f'The number of emails with at least one death mention: {count_email_at_least_1}')
print(f'The percentage of emails with at least one death mention: '
    f'{count_email_at_least_1 / len(processed_emails) * 100:.2f}%')    
print(f'Top 5 most frequent mentions with their counts:')
top_5 = top_5_death_mentions(death_stories)
for mention, count in top_5:
    print (f'{mention}: mentioned {count} times')

The total number of death mentions: 2726
The number of emails with at least one death mention: 816
The percentage of emails with at least one death mention: 61.35%
Top 5 most frequent mentions with their counts:
died: mentioned 767 times
death: mentioned 634 times
deceased: mentioned 344 times
late husband: mentioned 160 times
late father: mentioned 155 times


**4.4 Percentage splits.** Many emails offer a split of the money (e.g. "70% for us", "20% for you", "10% for expenses"). Find the most common split patterns.

Rules:
1. A **percentage** is a number (integer or decimal) directly followed by `%`, or followed by one space and `%`: `70%`, `20 %`, `2.5%`. Words such as `percent` do not count.
2. The **split** of an email: take all percentages in the email, remove `100%`, remove repeated values, and sort the values from largest to smallest.
3. An email **offers a split** only if this list has **at least 2 values** and the values add up to **exactly 100**.
4. Write a split as the values joined by ` - `, each followed by `%`. Write whole numbers without decimals (`70%`, not `70.0%`); keep real decimals (`2.5%`). Example: `70% - 20% - 10%`.

**Print:**
- the number of emails that offer a split;
- the **5 most common** split patterns with how often they appear, most common first (if two patterns are tied, the one that appears first in the file comes first), in this format:
```
70% - 20% - 10%: appears 15 times
75% - 20% - 5%: appears 12 times
...
```
*(These numbers are only an example of the format.)*

In [10]:
# YOUR CODE HERE
def emails_offer_split(percentage_splits):
    offer_splits = []
    for split in percentage_splits:
        if split and len(split) >= 2 and sum(split) == 100:
            offer_splits.append(split)
    return offer_splits

def sorted_percentage_split_count(offer_splits):
    count_splits = {}
    for split in offer_splits:
        split = ' - '.join((str(int(value)) if value.is_integer() else str(value)) + '%' 
                           for value in split
                          )
        count_splits[split] = count_splits.get(split, 0) + 1
    return sorted(count_splits.items(), key=lambda x: x[1], reverse=True)
        

regex = r'(\d+(?:\.\d+)?) ?%'
percentage_raws = [re.findall(regex,email) for email in processed_emails]

percentage_splits = []
for i in range (len(percentage_raws)):
    percentage_splits.append(sorted(
        {float(percentage_raws[i][j]) for j in range(len(percentage_raws[i])) 
         if float(percentage_raws[i][j]) != 100},
        reverse = True
    ))

offer_splits = emails_offer_split(percentage_splits)
sorted_count_splits = sorted_percentage_split_count(offer_splits)

print(f'The number of emails offering a split: {len(offer_splits)}')
print(f'Top 5 most common split patterns:')
for split, count in sorted_count_splits[:5]:
    print(f'{split}: appears {count} times')

The number of emails offering a split: 543
Top 5 most common split patterns:
60% - 30% - 10%: appears 106 times
70% - 25% - 5%: appears 84 times
70% - 20% - 10%: appears 56 times
75% - 20% - 5%: appears 45 times
60% - 35% - 5%: appears 39 times


**4.5 Scam score.** Give each email a score using your rules from 4.1–4.4:

| Component | Points | Rule |
|---|---|---|
| Urgency | 1 per occurrence | every occurrence of an urgency word (4.1 rules) |
| Money | 2 per mention | every money mention (4.2 rules) |
| Percentages | 1 per occurrence | every percentage (4.4 rule 1) — **including** `100%` and repeated values |
| Death | 1 (at most once) | 1 point if the email has at least one death mention (4.3 rules) |
| ALL CAPS | 1 (at most once) | split the text into words = runs of 2 or more letters `A–Z`/`a–z`. 1 point if **more than 20%** of these words are written entirely in UPPERCASE |

Number the emails **1, 2, 3, …** in the order they appear in the file.
Rank by score from highest to lowest; if two emails have the same score, the one that comes **first in the file** ranks higher.

**Print** the **top 10** emails, one per line: rank, email number, score and subject. For example:
```
1. email #57  score 25  SOME SUBJECT
```

In [11]:
# YOUR CODE HERE
def urgency_score(urgency):
    if urgency:
        return len(urgency)
    return 0

def money_score(money_mention):
    if money_mention:
        return len(money_mention)*2
    return 0

def percentage_score(percentage_raw):
    if percentage_raw:
        return len(percentage_raw)
    return 0

def death_score(death_story):
    return int(bool(death_story))

def all_cap_score(processed_email):
    regex = r'[A-Za-z]{2,}'
    email_words = re.findall(regex, processed_email)
    email_cap_words = [word for word in email_words if word.isupper()]
    if len(email_words) == 0:
        return 0
    return int((len(email_cap_words)/len(email_words)) > 0.2)

scam_scores = {i : urgency_score(urgency_detects[i]) 
               + money_score(money_mentions[i])
               + percentage_score(percentage_raws[i]) 
               + death_score(death_stories[i])
               + all_cap_score(processed_emails[i]) 
               for i in range(len(processed_emails))}

sorted_scam_scores = sorted(scam_scores.items(), key=lambda x: x[1], reverse=True)

print('Top 10 emails by score: ')
for rank, (idx, score) in enumerate(sorted_scam_scores[:10], start=1):
    print(f'{rank}. email #{idx+1}  score {score}  {decoded_emails[idx]["Subject"]}')

Top 10 emails by score: 
1. email #134  score 38  TREAT AS URGENT
2. email #591  score 32  Mutual Request
3. email #633  score 31  MUTAUL BUINESS RELATIONSHIP
4. email #634  score 31  MUTAUL BUINESS RELATIONSHIP
5. email #632  score 30  investment Relationship
6. email #244  score 29  INVESTMENT ASSISTANCE
7. email #394  score 29  INVESTMENT PROPOSAL
8. email #460  score 27  REPLY SOON
9. email #461  score 27  REPLY SOON
10. email #462  score 27  REPLY SOON


**4.6 Near-duplicates.** Scammers often send the same email again. Two emails are **near-duplicates** if **all three** conditions hold:

1. **Same sender address** — the email address inside the `From:` value (`name@domain.ext`), in lowercase.
   `"MR. JAMES NGOLA." <james_ngola2002@maktoob.com>` → `james_ngola2002@maktoob.com`
2. **Same normalized subject.** To normalize a subject:
   1. convert it to lowercase;
   2. remove any `re:`, `fw:` or `fwd:` at the start, repeatedly (spaces around them allowed), e.g. `RE: Fwd: URGENT` → `urgent`;
   3. remove every character that is not a letter, a digit or whitespace;
   4. replace each run of whitespace by a single space and strip both ends.
3. **Sent within 24 hours** of each other (a difference of at most exactly 24 hours counts).
   To get the time of an email, find in its `Date:` value the part that looks like `31 Oct 2002 02:38:20 +0000` (the day can have 1 or 2 digits) and convert it with
   `datetime.strptime(part, '%d %b %Y %H:%M:%S %z')` — the `+0000` time zone is then taken into account automatically.

**Skip** emails that have no sender address, an empty normalized subject, or no date of that form.

**Counting:** group the remaining emails by (sender address, normalized subject) and sort each group by time. An email is a **duplicate** if it was sent at most 24 hours after the **previous** email in its group. Example: 3 emails with the same sender and subject sent within one day → 2 duplicates.

**Print:**
- the number of emails that could be checked (not skipped);
- the number of duplicate emails;
- up to 5 examples of duplicate groups: sender address and normalized subject.

In [12]:
# YOUR CODE HERE
# print(decoded_emails[0].keys())
# print(repr(decoded_emails[0].get("From")))
# print(repr(decoded_emails[0].get("Subject")))
# print(repr(decoded_emails[0].get("Date")))
import datetime as dt

def get_sender_address(sender):
    regex = r'[\w.%+-]+@[\w.-]+\.[A-Za-z]{2,}'
    match = re.search(regex, sender)
    return match.group(0).lower() if match else ''

def normalize_subject(subject):
    subject = subject.lower()
    subject = re.sub(r'^\s*(?:(?:re|fw|fwd)\s*:\s*)+', '', subject)
    subject = re.sub(r'[^\w\s]|_', '', subject)
    subject = re.sub(r'\s+', ' ', subject).strip()
    return subject

def get_datetime(date):
    match = re.search(
        r'\b\d{1,2} [A-Za-z]{3} \d{4} '
        r'\d{2}:\d{2}:\d{2} [+-]\d{4}\b',
        date
    )
    if not match:
        raise ValueError('Không có ngày đúng định dạng')

    return dt.datetime.strptime(
        match.group(0), '%d %b %Y %H:%M:%S %z'
    )

checked = 0
counting_emails = {}
for email in decoded_emails:
    sender_address = get_sender_address(email['From'])
    subject_normalized = normalize_subject(email['Subject'])
    
    if not sender_address or not subject_normalized:
        continue
    
    try:
        send_datetime = get_datetime(email['Date'])
    except (ValueError, TypeError):
        continue
    
    checked+=1
    group = (sender_address, subject_normalized)
    if group not in counting_emails:
        counting_emails[group] = []

    counting_emails[group].append(send_datetime)

sorted_counting_emails = {group : sorted(sends_time) 
                          for group, sends_time in counting_emails.items()}
duplicates = 0
examples = []

for group, sends_time in sorted_counting_emails.items():
    group_duplicates = 0
    for previous, current in zip(sends_time, sends_time[1:]):
        if current - previous <= dt.timedelta(hours=24):
            group_duplicates += 1
    
    duplicates += group_duplicates
    if group_duplicates > 0 and len(examples) < 5:
        examples.append(group)

print(f'The number of emails that could be checked: {checked}')
print(f'The the number of duplicate emails: {duplicates}')
print('The 5 examples of duplicate groups: ')
for (sender, subject) in examples:
    print(f'- {sender}: {subject}')

The number of emails that could be checked: 941
The the number of duplicate emails: 210
The 5 examples of duplicate groups: 
- martinchime@borad.com: urgent
- kss07@go.com: attention
- davidmensoh@mail.com: urgent
- patrickmohammed@omaninfo.com: confidentialcooperation
- macontu028@yahoo.com: urgent business offer
